# 🕸️ Caso 4 — AgroTech Piracicaba: ¿quién conecta a la empresa?

> **Módulos del MBA relacionados:** Social Network Analysis I y II

## El caso
**AgroTech Piracicaba** es una startup de tecnología agrícola con 60 empleados. Creció rápido, y Luiza, su CEO, tiene dos miedos:
- "Los equipos no se hablan": cree que se formaron **silos** entre departamentos.
- "Si se va la persona equivocada, nos paramos": quiere saber quiénes son los **puentes** de la comunicación.

RR. HH. exportó (de forma anónima y agregada) la cantidad de **mensajes de Slack entre cada par de personas** en los últimos 3 meses. Te toca hacer un **análisis de red organizacional** (ONA).

## Tu misión: responder estas preguntas
1. ¿Qué tan conectada está la empresa? ¿Hay personas aisladas?
2. ¿Quiénes tienen más contactos (centralidad de grado)?
3. ¿Quiénes son los **puentes** entre equipos (intermediación, *betweenness*)?
4. ¿Quiénes están conectados con la gente más conectada (autovector, *eigenvector*)?
5. ¿Las **comunidades** que aparecen en la red coinciden con los departamentos? (silos)
6. ¿Qué departamentos hablan entre sí y cuáles no?
7. ¿Qué pasa con la red si el principal puente se va?

## Diccionario de datos
**funcionarios.csv**: `id` (F01…F60), `nome`, `departamento`, `cargo`, `tempo_empresa_anos`

**mensagens.csv**: `origem` (quién envió), `destino` (quién recibió), `qtd_mensagens` (cantidad en 3 meses)

## Conceptos clave
| concepto | en la red | en la empresa |
|---|---|---|
| **nodo** | un punto | una persona |
| **arista** (*edge*) | una línea | una relación (intercambiaron mensajes) |
| **peso** | grosor de la línea | cuántos mensajes |
| **grado** | cantidad de conexiones | a cuántas personas distintas habla |
| **intermediación** | cuántos caminos cortos pasan por el nodo | quién hace de puente |
| **comunidad** | grupo muy conectado por dentro | equipos reales (formales o no) |

## Qué vas a practicar
`networkx`: crear grafos desde pandas, grafo dirigido y no dirigido · densidad, componentes, distancias · centralidades · comunidades (Louvain) y modularidad · `crosstab` · `pivot_table` · visualización de redes · simulación "¿y si…?"

## Paso 0 — Genera la base de datos

Este código simula la red de mensajes de la empresa y guarda `funcionarios.csv` y `mensagens.csv`. Ejecútalo una vez.

In [ ]:
# ============================================================
# PASO 0 — GENERADOR DE DATOS (ejecútalo una vez)
# ============================================================
import numpy as np
import pandas as pd
from itertools import combinations

rng = np.random.default_rng(11)

deptos = {"Diretoria": 2, "Engenharia": 14, "Dados": 9, "Produto": 7,
          "Comercial": 12, "Marketing": 8, "Pessoas & Finanças": 8}
nomes = ["Ana", "Bruno", "Carla", "Diego", "Eduarda", "Felipe", "Gabriela", "Henrique", "Isabela",
         "João", "Larissa", "Marcos", "Natália", "Otávio", "Paula", "Rafael", "Sofia", "Thiago",
         "Vanessa", "Yuri", "Beatriz", "Caio", "Débora", "Enzo", "Fernanda", "Gustavo", "Helena",
         "Igor", "Júlia", "Leonardo"]
sobrenomes = ["Silva", "Santos", "Lima", "Costa", "Rocha", "Alves", "Melo", "Prado", "Dias", "Ramos"]
pares = [f"{a} {b}" for a in nomes for b in sobrenomes]
escolhidos = rng.choice(pares, size=sum(deptos.values()), replace=False)

cargos_depto = {
    "Diretoria": ["CEO", "COO"],
    "Engenharia": ["Gerente de Engenharia", "Tech Lead"] + ["Engenheiro(a) de Software"] * 12,
    "Dados": ["Head de Dados"] + ["Cientista de Dados"] * 4 + ["Analista de Dados"] * 4,
    "Produto": ["Head de Produto", "Product Manager", "Product Manager"] + ["Designer UX"] * 4,
    "Comercial": ["Gerente Comercial"] + ["Executivo(a) de Vendas"] * 8 + ["Customer Success"] * 3,
    "Marketing": ["Gerente de Marketing"] + ["Analista de Marketing"] * 7,
    "Pessoas & Finanças": ["Gerente de Pessoas", "Controller"] + ["Analista"] * 6,
}
linhas = []
for depto, qtd in deptos.items():
    for cargo in cargos_depto[depto]:
        linhas.append({"departamento": depto, "cargo": cargo})
funcionarios = pd.DataFrame(linhas)
funcionarios.insert(0, "id", [f"F{i:02d}" for i in range(1, len(funcionarios) + 1)])
funcionarios.insert(1, "nome", escolhidos)
funcionarios["tempo_empresa_anos"] = rng.uniform(0.3, 8, len(funcionarios)).round(1)

# Personas "especiales" de la historia
pm = funcionarios.index[funcionarios["cargo"] == "Product Manager"][0]      # puente Producto ↔ todos
analista = funcionarios.index[funcionarios["cargo"] == "Analista de Dados"][0]  # puente Datos ↔ Negocio
recem = funcionarios.index[funcionarios["cargo"] == "Analista de Dados"][-1]    # llegó hace poco
estagio = funcionarios.index[funcionarios["departamento"] == "Marketing"][-1]     # llegó la semana pasada
funcionarios.loc[recem, "tempo_empresa_anos"] = 0.2
funcionarios.loc[funcionarios["cargo"] == "CEO", "nome"] = "Luiza Andrade"
funcionarios.loc[estagio, ["cargo", "tempo_empresa_anos"]] = ["Estagiário(a) de Marketing", 0.0]

prob_entre = {frozenset(["Engenharia", "Produto"]): 0.08, frozenset(["Dados", "Engenharia"]): 0.05,
              frozenset(["Comercial", "Marketing"]): 0.10, frozenset(["Dados", "Produto"]): 0.03}
dep = funcionarios["departamento"].to_numpy()

registros = []
for i, j in combinations(range(len(funcionarios)), 2):
    if estagio in (i, j):
        continue
    if dep[i] == dep[j]:
        p, w = 0.45, 20
    elif "Diretoria" in (dep[i], dep[j]):
        p, w = 0.25, 8
    else:
        p, w = prob_entre.get(frozenset([dep[i], dep[j]]), 0.004), 6
    if pm in (i, j) and {dep[i], dep[j]} & {"Engenharia", "Dados", "Comercial", "Marketing"}:
        p, w = 0.55, 12
    if analista in (i, j) and {dep[i], dep[j]} & {"Comercial", "Marketing"}:
        p, w = 0.45, 10
    if recem in (i, j):
        p = 1.0 if {i, j} == {recem, funcionarios.index[funcionarios["cargo"] == "Head de Dados"][0]} else 0
    if rng.random() < p:
        registros.append((i, j, max(1, rng.poisson(w))))
        if rng.random() < 0.9:
            registros.append((j, i, max(1, rng.poisson(w * rng.uniform(0.5, 1.5)))))

ids = funcionarios["id"].to_numpy()
mensagens = pd.DataFrame([(ids[a], ids[b], q) for a, b, q in registros],
                         columns=["origem", "destino", "qtd_mensagens"])

funcionarios.to_csv("funcionarios.csv", index=False)
mensagens.to_csv("mensagens.csv", index=False)
print(f"✅ Archivos creados: funcionarios.csv ({len(funcionarios)} personas) y mensagens.csv ({len(mensagens)} filas)")

---
# ✍️ Parte A — Tu turno: el paso a paso en comentarios

Cada celda de abajo tiene **solo comentarios** con lo que debes hacer y algunas pistas.
Escribe tu código debajo de `# ✍️ Tu código aquí:` y ejecútalo.

- Si no conoces una función, búscala ("pandas como convertir texto a fecha"): buscar es parte del trabajo de un analista.
- Equivocarse es normal. Lee el mensaje de error de abajo hacia arriba: la última línea dice qué pasó.
- Si una celda que modifica datos se ejecuta dos veces, puede dar error. Vuelve a ejecutar desde el paso donde lees los archivos.

### Paso 1 — Importar las librerías

**networkx** es la librería estándar de Python para redes. En Colab ya viene instalada.

In [ ]:
# PASO 1 — IMPORTAR LAS LIBRERÍAS
# 1. Importa pandas (pd), numpy (np), networkx (nx) y matplotlib.pyplot (plt)
# 2. Importa Line2D desde matplotlib.lines (servirá para la leyenda del gráfico de red)
# 3. Define una lista de colores fijos, uno por departamento (7 departamentos)

# ✍️ Tu código aquí:


### Paso 2 — Importar y conocer los datos

Una red se guarda casi siempre como dos tablas: **nodos** (personas) y **aristas** (relaciones).

In [ ]:
# PASO 2 — IMPORTAR Y CONOCER LOS DATOS
# 1. Lee "funcionarios.csv" y "mensagens.csv"
# 2. ¿Cuántas personas hay por departamento? (.value_counts())
# 3. ¿Cuántas filas tiene mensagens? ¿Cuál es el total de mensajes?
# 4. Crea dos "diccionarios" para traducir id → nombre e id → departamento:
#    nome = funcionarios.set_index("id")["nome"]

# ✍️ Tu código aquí:


### Paso 3 — Construir el grafo dirigido

En un grafo **dirigido** importa el sentido: Ana → Bruno no es lo mismo que Bruno → Ana.

In [ ]:
# PASO 3 — CONSTRUIR EL GRAFO DIRIGIDO
# 1. Crea el grafo con nx.from_pandas_edgelist(mensagens, source="origem", target="destino",
#    edge_attr="qtd_mensagens", create_using=nx.DiGraph) → D
# 2. ⚠️ Agrega TODOS los nodos con D.add_nodes_from(funcionarios["id"])
#    (si alguien no envió ni recibió mensajes, ¡no aparece en la tabla de aristas!)
# 3. Imprime D para ver cuántos nodos y aristas tiene
# 4. ¿Quién RECIBE más mensajes? → D.in_degree(weight="qtd_mensagens")
#    ¿Quién ENVÍA más? → D.out_degree(weight="qtd_mensagens")

# ✍️ Tu código aquí:


### Paso 4 — Grafo no dirigido con pesos

Para medir **relaciones** no importa quién escribió primero: suma los mensajes en los dos sentidos.

In [ ]:
# PASO 4 — GRAFO NO DIRIGIDO CON PESOS
# 1. Crea en una copia de mensagens las columnas a y b con el par ordenado (el id menor primero),
#    para que (F01, F02) y (F02, F01) sean el mismo par
#    💡 Pista: menor = m["origem"] < m["destino"]; m["a"] = m["origem"].where(menor, m["destino"])
# 2. Agrupa por (a, b) y suma qtd_mensagens → pares
# 3. Crea distancia = 1 / qtd_mensagens (más mensajes = más "cerca"; se usa en los caminos más cortos)
# 4. Construye G = nx.from_pandas_edgelist(pares, "a", "b", edge_attr=["qtd_mensagens", "distancia"])
# 5. Agrega todos los nodos otra vez y guarda el departamento como atributo de cada nodo
#    (nx.set_node_attributes)

# ✍️ Tu código aquí:


### Paso 5 — Métricas generales de la red

Primero la foto general: tamaño, densidad, conectividad y distancia media entre personas.

In [ ]:
# PASO 5 — MÉTRICAS GENERALES DE LA RED
# 1. Imprime número de nodos y de aristas
# 2. Densidad: nx.density(G) → % de todas las relaciones posibles que existen
# 3. Componentes conectados: list(nx.connected_components(G)) → ¿cuántos hay?
# 4. Aislados: list(nx.isolates(G)) → muestra sus nombres
# 5. En el componente más grande, calcula la distancia media en "saltos":
#    nx.average_shortest_path_length(G.subgraph(componente_mayor))
# 6. Coeficiente de agrupamiento: nx.average_clustering(G)
#    (¿los amigos de mis amigos también son mis amigos?)

# ✍️ Tu código aquí:


### Paso 6 — Centralidades: ¿quién importa y por qué?

Cada centralidad responde una pregunta distinta. No existe la persona más importante a secas, hay que decir **en qué sentido**.

In [ ]:
# PASO 6 — CENTRALIDADES: ¿QUIÉN IMPORTA Y POR QUÉ?
# Crea un DataFrame cent con una columna por métrica (cada función devuelve un dict {nodo: valor}):
# 1. grau = dict(G.degree())                                    → ¿con cuántas personas habla?
# 2. forca = dict(G.degree(weight="qtd_mensagens"))             → ¿cuántos mensajes en total?
# 3. intermediacao = nx.betweenness_centrality(G, weight="distancia") → ¿es un puente?
#    ⚠️ Aquí el peso debe ser una DISTANCIA (por eso creamos distancia = 1/mensajes)
# 4. proximidade = nx.closeness_centrality(G, distance="distancia")  → ¿llega rápido a todos?
# 5. autovetor = nx.eigenvector_centrality(G, weight="qtd_mensagens", max_iter=1000)
#    → ¿está conectado con gente bien conectada?
# 6. Une con nombre, departamento y cargo, y muestra el top 5 de grau, intermediacao y autovetor

# ✍️ Tu código aquí:


### Paso 7 — Comunidades: ¿hay silos?

El algoritmo de **Louvain** busca grupos de personas que se hablan mucho entre sí y poco con el resto. Después se comparan con los departamentos formales.

In [ ]:
# PASO 7 — COMUNIDADES: ¿HAY SILOS?
# 1. Detecta comunidades: nx.community.louvain_communities(G, weight="qtd_mensagens", seed=42)
#    (devuelve una lista de conjuntos de nodos)
# 2. Calcula la modularidad: nx.community.modularity(G, comunidades, weight="qtd_mensagens")
#    (cerca de 0 = sin estructura; por encima de 0.3 = comunidades bien marcadas)
# 3. Crea un dict {persona: número_de_comunidad} y agrégalo como columna en cent
# 4. Compara con pd.crosstab(cent["departamento"], cent["comunidade"])
#    🤔 ¿Cada comunidad es un departamento? ¿Qué departamentos quedaron juntos?

# ✍️ Tu código aquí:


### Paso 8 — ¿Qué departamentos hablan entre sí?

Lleva la red al nivel de departamentos: una **matriz** de mensajes de origen × destino.

In [ ]:
# PASO 8 — ¿QUÉ DEPARTAMENTOS HABLAN ENTRE SÍ?
# 1. Agrega a mensagens las columnas depto_origem y depto_destino (usa .map(depto))
# 2. Matriz: pivot_table(index="depto_origem", columns="depto_destino", values="qtd_mensagens",
#    aggfunc="sum", fill_value=0)
# 3. Para cada departamento: % de sus mensajes que quedan DENTRO del propio departamento
#    💡 Pista: np.diag(matriz) / matriz.sum(axis=1)
# 4. Dibuja la matriz como mapa de calor (plt.imshow) con una sola tonalidad (azules)

# ✍️ Tu código aquí:


### Paso 9 — Visualizar la red

Dibuja la red: color = departamento, tamaño = intermediación (los puentes se ven grandes) y nombres solo para los 5 puentes principales.

In [ ]:
# PASO 9 — VISUALIZAR LA RED
# 1. Calcula posiciones con nx.spring_layout(..., weight="qtd_mensagens", seed=7)
#    (los nodos que se hablan mucho quedan cerca)
#    💡 Calcula el layout solo del componente principal y pon a los aislados en una esquina
# 2. Dibuja las aristas con nx.draw_networkx_edges (finas y transparentes: alpha=0.15)
# 3. Dibuja los nodos con nx.draw_networkx_nodes:
#    - node_color = color del departamento de cada nodo
#    - node_size = 80 + 3000 * intermediación
# 4. Escribe el nombre solo de los 5 nodos con mayor intermediación (nx.draw_networkx_labels)
# 5. Agrega una leyenda de departamentos, quita los ejes (plt.axis("off")) y guarda en PNG

# ✍️ Tu código aquí:


### Paso 10 — Simulación: ¿y si el principal puente se va?

Saca de la red a la persona con mayor intermediación y mide cuánto empeora la comunicación.

In [ ]:
# PASO 10 — SIMULACIÓN: ¿Y SI EL PRINCIPAL PUENTE SE VA?
# 1. Identifica al puente principal: cent["intermediacao"].idxmax()
# 2. Copia el grafo (G2 = G.copy()) y quita ese nodo (G2.remove_node(...))
# 3. Compara antes/después:
#    - eficiencia global: nx.global_efficiency(G) → 1 = todos a un salto
#    - distancia media del componente principal
#    - % de pares de departamentos distintos que siguen teniendo contacto directo
# 4. Escribe en un comentario: ¿qué le recomendarías a la CEO?

# ✍️ Tu código aquí:


---
# 🔑 Parte B — Solución comentada

**Intenta resolver la Parte A antes de mirar esto.** Después compara: no hay una única forma correcta,
pero fíjate en las funciones que usé y en los comentarios que explican el porqué.

> Ejecuta la Parte B **en orden, desde el Paso 1**: vuelve a leer los archivos originales, así que no depende de lo que hiciste en la Parte A.

### ✅ Paso 1 — Importar las librerías

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Un color fijo por departamento (siempre el mismo en todos los gráficos)
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})

### ✅ Paso 2 — Importar y conocer los datos

In [ ]:
funcionarios = pd.read_csv("funcionarios.csv")
mensagens = pd.read_csv("mensagens.csv")

print(funcionarios["departamento"].value_counts())
print(f"\nFilas de mensajes: {len(mensagens)} | total de mensajes: {mensagens['qtd_mensagens'].sum()}")
display(mensagens.head())

nome = funcionarios.set_index("id")["nome"]           # id → nombre
depto = funcionarios.set_index("id")["departamento"]  # id → departamento

### ✅ Paso 3 — Construir el grafo dirigido

In [ ]:
D = nx.from_pandas_edgelist(mensagens, source="origem", target="destino",
                            edge_attr="qtd_mensagens", create_using=nx.DiGraph)
print("Antes de agregar todos los nodos:", D.number_of_nodes(), "personas")
D.add_nodes_from(funcionarios["id"])  # incluye a quien no tiene ningún mensaje
print(D)

recebidas = pd.Series(dict(D.in_degree(weight="qtd_mensagens")), name="recebidas")
enviadas = pd.Series(dict(D.out_degree(weight="qtd_mensagens")), name="enviadas")
fluxo = pd.concat([recebidas, enviadas], axis=1)
fluxo["nome"] = fluxo.index.map(nome)
display(fluxo.sort_values("recebidas", ascending=False).head(5))

### ✅ Paso 4 — Grafo no dirigido con pesos

In [ ]:
m = mensagens.copy()
menor = m["origem"] < m["destino"]
m["a"] = m["origem"].where(menor, m["destino"])
m["b"] = m["destino"].where(menor, m["origem"])
pares = m.groupby(["a", "b"], as_index=False)["qtd_mensagens"].sum()
pares["distancia"] = 1 / pares["qtd_mensagens"]  # más mensajes = relación más "corta"

G = nx.from_pandas_edgelist(pares, "a", "b", edge_attr=["qtd_mensagens", "distancia"])
G.add_nodes_from(funcionarios["id"])
nx.set_node_attributes(G, depto.to_dict(), "departamento")
print(G)

💡 **¿Por qué no usar `D.to_undirected()`?** Porque, cuando existen las dos direcciones, se queda con el peso de una sola en lugar de sumarlas. Detalles así cambian el resultado sin dar ningún error. Siempre conviene entender qué hace cada función.

### ✅ Paso 5 — Métricas generales de la red

In [ ]:
print("Personas:", G.number_of_nodes(), "| Relaciones:", G.number_of_edges())
print(f"Densidad: {nx.density(G):.3f} → existe el {nx.density(G):.1%} de las relaciones posibles")

componentes = list(nx.connected_components(G))
print("Componentes conectados:", len(componentes))
aislados = list(nx.isolates(G))
print("Aislados:", [f"{nome[i]} ({funcionarios.set_index('id').loc[i, 'cargo']})" for i in aislados])

maior = G.subgraph(max(componentes, key=len))
print(f"Distancia media en el componente principal: {nx.average_shortest_path_length(maior):.2f} saltos")
print(f"Coeficiente de agrupamiento medio: {nx.average_clustering(G):.2f}")

### ✅ Paso 6 — Centralidades: ¿quién importa y por qué?

In [ ]:
cent = pd.DataFrame({
    "grau": dict(G.degree()),
    "forca": dict(G.degree(weight="qtd_mensagens")),
    "intermediacao": nx.betweenness_centrality(G, weight="distancia"),
    "proximidade": nx.closeness_centrality(G, distance="distancia"),
    "autovetor": nx.eigenvector_centrality(G, weight="qtd_mensagens", max_iter=1000),
})
cent = funcionarios.set_index("id")[["nome", "departamento", "cargo"]].join(cent)

for metrica in ["grau", "intermediacao", "autovetor"]:
    print(f"\n🏆 Top 5 — {metrica}")
    display(cent.sort_values(metrica, ascending=False)
            .head(5)[["nome", "departamento", "cargo", metrica]].round(3))

💡 **Cómo leer las centralidades:** quien tiene **grado** alto conoce a mucha gente. Quien tiene **intermediación** alta es un puente: une grupos que, sin esa persona, casi no se hablan (y por eso su salida es un riesgo). Quien tiene **autovector** alto está cerca del "núcleo de poder". Fíjate que en el top de intermediación aparecen cargos que no son de jefatura: la red informal no copia el organigrama.

### ✅ Paso 7 — Comunidades: ¿hay silos?

In [ ]:
comunidades = nx.community.louvain_communities(G, weight="qtd_mensagens", seed=42)
print(f"Comunidades encontradas: {len(comunidades)}")
print(f"Modularidad: {nx.community.modularity(G, comunidades, weight='qtd_mensagens'):.2f}")

mapa = {pessoa: i + 1 for i, grupo in enumerate(comunidades) for pessoa in grupo}
cent["comunidade"] = cent.index.map(mapa)
display(pd.crosstab(cent["departamento"], cent["comunidade"]))

### ✅ Paso 8 — ¿Qué departamentos hablan entre sí?

In [ ]:
m = mensagens.assign(depto_origem=mensagens["origem"].map(depto),
                     depto_destino=mensagens["destino"].map(depto))
matriz = m.pivot_table(index="depto_origem", columns="depto_destino", values="qtd_mensagens",
                       aggfunc="sum", fill_value=0)

interno = pd.Series(np.diag(matriz) / matriz.sum(axis=1), index=matriz.index)
print("% de mensajes que se quedan dentro del propio departamento:")
print((interno * 100).round(0).sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(matriz.values, cmap=plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "azuis", ["#eef4fc", "#3987e5", "#0d366b"]))
ax.set_xticks(range(len(matriz.columns)), matriz.columns, rotation=40, ha="right")
ax.set_yticks(range(len(matriz.index)), matriz.index)
for i in range(len(matriz.index)):
    for j in range(len(matriz.columns)):
        v = matriz.values[i, j]
        ax.text(j, i, v, ha="center", va="center", fontsize=9,
                color="white" if v > matriz.values.max() * 0.5 else "#0b0b0b")
ax.set_xlabel("Departamento que recibe")
ax.set_ylabel("Departamento que envía")
ax.set_title("Mensajes entre departamentos (3 meses)")
fig.colorbar(im, ax=ax, shrink=0.8, label="mensajes")
plt.tight_layout()
plt.show()

### ✅ Paso 9 — Visualizar la red

In [ ]:
lista_deptos = list(funcionarios["departamento"].unique())
cor = dict(zip(lista_deptos, PALETA))
nos = list(G.nodes)

# Posiciones: calculamos el layout solo del componente principal (los aislados "empujarían" todo
# hacia un rincón) y ponemos a los aislados en una esquina
nucleo = max(nx.connected_components(G), key=len)
pos = nx.spring_layout(G.subgraph(nucleo), weight="qtd_mensagens", seed=7, k=0.6, iterations=300)
for i, n in enumerate(nx.isolates(G)):
    pos[n] = np.array([1.05, -1.0 + 0.12 * i])

plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G, pos, alpha=0.15, edge_color="#52514e",
                       width=[0.4 + d["qtd_mensagens"] / 40 for _, _, d in G.edges(data=True)])
nx.draw_networkx_nodes(G, pos, nodelist=nos,
                       node_color=[cor[depto[n]] for n in nos],
                       node_size=[80 + 3000 * cent.loc[n, "intermediacao"] for n in nos],
                       edgecolors="#fcfcfb", linewidths=2)
top5 = cent.nlargest(5, "intermediacao").index
nx.draw_networkx_labels(G, pos, labels={n: nome[n] for n in top5}, font_size=10, font_weight="bold",
                        bbox=dict(facecolor="#fcfcfb", edgecolor="none", alpha=0.8, pad=1.5))

legenda = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=11, label=d)
           for d, c in cor.items()]
for n in nx.isolates(G):
    plt.annotate(f"aislado: {nome[n]}", pos[n], xytext=(-10, 10), textcoords="offset points",
                 ha="right", fontsize=9, color="#52514e")
plt.legend(handles=legenda, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False, fontsize=10)
plt.title("Red de comunicación de AgroTech · tamaño = intermediación (puentes)", fontsize=14)
plt.axis("off")
plt.savefig("rede_agrotech.png", dpi=150, bbox_inches="tight")
plt.show()

### ✅ Paso 10 — Simulación: ¿y si el principal puente se va?

In [ ]:
puente = cent["intermediacao"].idxmax()
G2 = G.copy()
G2.remove_node(puente)

def distancia_media(grafo):
    nucleo = grafo.subgraph(max(nx.connected_components(grafo), key=len))
    return nx.average_shortest_path_length(nucleo)

def pares_deptos_conectados(grafo):
    pares_d = {frozenset([depto[u], depto[v]]) for u, v in grafo.edges if depto[u] != depto[v]}
    total = len(lista_deptos) * (len(lista_deptos) - 1) / 2
    return len(pares_d) / total

print(f"Si {nome[puente]} ({cent.loc[puente, 'cargo']}, {cent.loc[puente, 'departamento']}) se va:")
print(f"  Eficiencia global:        {nx.global_efficiency(G):.3f} → {nx.global_efficiency(G2):.3f}")
print(f"  Distancia media (saltos): {distancia_media(G):.2f} → {distancia_media(G2):.2f}")
print(f"  Pares de deptos con contacto directo: {pares_deptos_conectados(G):.0%} → {pares_deptos_conectados(G2):.0%}")

💡 **Recomendaciones típicas de un ONA:** (1) reconocer y retener a los puentes, y repartir su carga (si una sola persona conecta Producto con Comercial, eso es frágil); (2) crear rituales entre los departamentos aislados, como demos quincenales o un canal compartido; (3) un plan de integración para quien está aislado (quien hace la pasantía en Marketing todavía no intercambió ningún mensaje).

---
## 🎤 Lo que le dirías a Luiza (CEO)
1. **Sí hay silos:** la mayoría de los mensajes se quedan dentro de cada departamento, y las comunidades que detectó el algoritmo reproducen casi exactamente el organigrama.
2. **Pocos puentes sostienen la empresa:** quien ocupa el puesto de Product Manager tiene, de lejos, la mayor intermediación; después viene la propia Luiza. Si esa persona de Producto se va, los pares de departamentos con contacto directo caen de 71% a 57%.
3. **Aislamiento:** quien hace la pasantía en Marketing no aparece en ningún mensaje, y la persona que entró hace poco en Datos solo habla con quien dirige su área.
4. **Acciones:** retener y "duplicar" a los puentes, crear espacios entre Datos y Comercial, y un buddy para quien acaba de llegar.

## 🚀 Retos extra
1. Practica con una red clásica que ya viene en networkx: `nx.karate_club_graph()`. Es el club de karate que se dividió en dos: ¿Louvain encuentra la división real? (atributo `club` de cada nodo)
2. Haz el gráfico interactivo con **pyvis** (`!pip install pyvis`) y ábrelo en el navegador.
3. Dibuja la **red ego** de un puente: `nx.ego_graph(G, puente, radius=1)`.
4. Compara recibidas vs. enviadas: ¿quién recibe mucho más de lo que envía? (¿cuello de botella o referente?)